# 14-01 从状态转移训练到连续预测

本实验对应第 14 章，按“完整轨迹 → 相邻训练样本 → 单步训练 → 连续预测”的顺序运行。依赖 NumPy、Matplotlib 和 PyTorch，CPU 即可。

运行过程中请始终区分环境时间和训练更新次数。实验会分别报告接触样本与未接触样本的误差，并把训练好的网络放回同一状态、不同动作的测试中。图内使用英文以避免在线环境缺少中文字体；每个代码单元后都给出中文结果说明。


## 1. 先生成完整轨迹，再划分训练与测试

位置单位为米，每个动作表示一个周期内推杆的位移。每条轨迹有 24 个动作、25 个状态。先划分轨迹编号，再抽出相邻状态，保证测试集没有混入训练轨迹的下一帧。第一幅图中两条线分别是推杆与物体；不要期待它们永远同时移动。

In [ ]:
import numpy as np  # 用数组保存轨迹与动作
import matplotlib.pyplot as plt  # 绘制位置曲线
import torch  # 使用神经网络训练与梯度计算
from torch import nn  # 使用线性层和激活函数
torch.set_num_threads(1)  # 限制线程数以便小模型在普通处理器上稳定运行
torch.manual_seed(14)  # 固定模型初始化与训练采样随机性
rng = np.random.default_rng(14)  # 固定教学数据的随机数种子
gap = 0.025  # 设定接触时推杆中心与物体中心的最小间距
def transition(state, action):  # 定义只用于生成答案的教学接触规则
    p = state[0] + float(action)  # 根据实际位移更新推杆位置
    b = max(state[1], p + gap)  # 接触时推动物体向右移动而不产生吸附
    return np.array([p, b], dtype=np.float32)  # 返回下一时刻两个位置
episodes = []  # 保存完整轨迹以便按轨迹划分数据
for episode in range(100):  # 生成一百条不同起点与动作的轨迹
    p = rng.uniform(0.05, 0.3)  # 随机选择推杆初始位置
    states = [np.array([p, p + rng.uniform(0.04, 0.16)], dtype=np.float32)]  # 将物体放在推杆右侧
    actions = rng.uniform(-0.015, 0.025, size=24).astype(np.float32)  # 生成靠近与离开的位移命令
    for action in actions:  # 依次执行当前轨迹的全部动作
        states.append(transition(states[-1], action))  # 记录动作之后的真实下一状态
    episodes.append((np.stack(states), actions))  # 将状态序列与区间动作一起保存
fig, ax = plt.subplots(figsize=(7, 3))  # 为一条真实交互创建简洁画布
ax.plot(episodes[0][0][:, 0], label="Pusher")  # 显示推杆随时间变化的位置
ax.plot(episodes[0][0][:, 1], label="Object")  # 显示物体随时间变化的位置
ax.set(xlabel="Step", ylabel="Position (m)", title="One recorded episode")  # 明确时间轴和物理单位
ax.legend()  # 标明两条曲线的含义
plt.show()  # 显示真实轨迹图


### 怎样读这张图

横轴是环境步数，不是训练轮数；纵轴是位置。推杆变化而物体线段水平，说明还没有接触，或推杆正在离开。进入接触后，物体才跟着向右移动，两者维持规定间距。图中还没有网络预测，不能据此判断模型是否学会。

状态点比动作多一个。动作编号 0 把状态 0 变成状态 1，而不是状态 1 变成状态 2。

## 2. 用四次观测核对三条训练样本

在上面的教学接触规则下，固定起点为推杆 0.20 m、物体 0.25 m，依次执行 +0.01、+0.04、−0.01 m。每次只把执行前状态和当次动作作为输入；执行后的状态用于计算答案。下面直接打印三个输入 $x=(p_t,b_t,a_t)$ 和位移答案 $y=s_{t+1}-s_t$。这组数与正文图 14-2 相同，尚未经过神经网络。

In [ ]:
example_states = [np.array([0.20, 0.25], dtype=np.float32)]  # 固定与正文图示相同的执行前位置
example_actions = np.array([0.01, 0.04, -0.01], dtype=np.float32)  # 指定三个区间内实际采用的动作
for action in example_actions:  # 依次执行记录中的三个动作
    example_states.append(transition(example_states[-1], action))  # 保存动作之后才得到的下一状态
for index, action in enumerate(example_actions):  # 从四个状态中逐对取出三个样本
    before = example_states[index]  # 取本次动作之前实际测到的状态
    after = example_states[index + 1]  # 取本次动作之后实际测到的状态
    inputs = np.r_[before, action]  # 将执行前状态与区间动作组成三维输入
    target = after - before  # 将下一状态转成两维变化量监督答案
    print(f"样本 {index + 1}：输入 {np.round(inputs, 3)}；变化量答案 {np.round(target, 3)}")  # 用米打印一条可核对的训练样本


### 怎样核对打印结果

样本 1 的输入应为 $[0.20,0.25,0.01]$，答案为 $[0.01,0]$：推杆靠近，物体不动。样本 2 的输入从**上一次真实执行后的状态**开始，即 $[0.21,0.25,0.04]$；答案为 $[0.04,0.025]$，说明物体被推动。样本 3 向左离开，物体位移再次为零。

这里打印的是记录拆出的训练答案，不是模型预测。后面的网络会试着从这三个输入数预测最后两个位移数；真正训练时还会使用前面生成的很多完整轨迹。

## 3. 用拆出的样本训练网络预测变化量

输入是推杆位置、物体位置和本周期动作三个数；输出是两个位置的变化量。先按完整轨迹将前 80 条划为训练集、后 20 条划为测试集，再分别拆样本，避免同一轨迹的相邻片段同时出现在两边。

输入均值与尺度只由训练轨迹计算，变化量统一除以 0.025 m。运行后先看训练曲线是否稳定下降；下一单元再把测试结果恢复为毫米，并和“物体永远不动”的规则比较。


In [ ]:
def make_pairs(items):  # 将若干完整轨迹整理为状态转移样本
    x = np.concatenate([np.column_stack((s[:-1], a)) for s, a in items])  # 拼接执行前状态与对应区间动作
    y = np.concatenate([np.diff(s, axis=0) for s, a in items])  # 由相邻状态计算真实变化量
    return x, y  # 返回输入与监督答案
train_x, train_y = make_pairs(episodes[:80])  # 只从训练轨迹构造训练样本
test_x, test_y = make_pairs(episodes[80:])  # 独立构造未见轨迹的测试样本
mean = train_x.mean(axis=0)  # 使用训练数据计算输入中心
scale = train_x.std(axis=0) + 1e-6  # 使用训练数据计算尺度并防止除零
x_train = torch.tensor((train_x - mean) / scale)  # 将归一化输入转成张量
y_train = torch.tensor(train_y / 0.025)  # 将变化量缩放到适合训练的量级
model = nn.Sequential(nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))  # 建立变化量预测网络
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # 设置更新网络权重的优化器
losses = []  # 保存实际计算出的训练误差
for step in range(700):  # 执行多次随机小批量参数更新
    ids = torch.randint(len(x_train), (256,))  # 从训练数据随机抽取当前批次
    prediction = model(x_train[ids])  # 预测两个位置的变化量
    loss = ((prediction - y_train[ids]) ** 2).mean()  # 比较预测变化与真实变化
    optimizer.zero_grad()  # 清除上一次更新留下的梯度
    loss.backward()  # 计算当前误差对参数的影响
    optimizer.step()  # 更新网络参数
    losses.append(loss.item())  # 保存本次训练得到的误差
fig, ax = plt.subplots(figsize=(7, 3))  # 创建训练曲线画布
ax.plot(losses)  # 绘制每次更新的真实误差
ax.set(xlabel="Parameter update", ylabel="Scaled MSE", yscale="log")  # 使用对数轴看清后期误差变化
plt.show()  # 显示训练过程


### 怎样读训练曲线

横轴是参数更新次数，与上一图的环境步数不同。纵轴是缩放后的均方误差，不能直接读成厘米。整体下降说明模型更能拟合训练样本；局部波动来自每次抽到的样本不同。

低训练损失不证明连续预测可靠。下面固定一段新动作，分别每步重新给真实状态、以及一直使用自己的预测，两者共用同一个网络。

In [ ]:
def predict_delta(inputs):  # 定义不再更新参数的变化量预测函数
    with torch.no_grad():  # 评测时关闭梯度记录
        values = torch.tensor((np.asarray(inputs, dtype=np.float32) - mean) / scale)  # 使用训练阶段固定的归一化量
        return model(values).numpy() * 0.025  # 把预测变化恢复成米
test_delta = predict_delta(test_x)  # 计算测试样本的单步变化预测
object_mae = np.abs(test_delta[:, 1] - test_y[:, 1]).mean()  # 计算物体位置单步平均绝对误差
baseline_mae = np.abs(test_y[:, 1]).mean()  # 计算物体永远不动的基线误差
contact_mask = test_y[:, 1] > 1e-7  # 按真实物体是否移动区分接触与未接触样本
print(f"测试样本：接触 {contact_mask.sum()} 条，未接触 {(~contact_mask).sum()} 条")  # 说明两类平均误差各包含多少样本
print(f"模型物体位移 MAE：全部 {object_mae * 1000:.2f} mm；接触 {np.abs(test_delta[contact_mask, 1] - test_y[contact_mask, 1]).mean() * 1000:.2f} mm；未接触 {np.abs(test_delta[~contact_mask, 1] - test_y[~contact_mask, 1]).mean() * 1000:.2f} mm")  # 分情况报告模型误差
print(f"不动基线 MAE：全部 {baseline_mae * 1000:.2f} mm；接触 {np.abs(test_y[contact_mask, 1]).mean() * 1000:.2f} mm")  # 报告简单规则在真正发生推动时的误差
same_state = np.array([0.21, 0.25], dtype=np.float32)  # 固定一个用于动作条件检查的当前状态
for probe_action in (-0.015, 0.010, 0.040):  # 依次测试向左、小幅向右和大幅向右动作
    probe_delta = predict_delta(np.r_[same_state, probe_action])  # 预测同一状态下不同动作的后果
    print(f"动作 {probe_action:+.3f} m → 推杆变化 {probe_delta[0]:+.3f} m，物体变化 {probe_delta[1]:+.3f} m")  # 检查输出是否随动作合理改变
actions = np.full(30, 0.012, dtype=np.float32)  # 规定一段连续向右的未见动作
truth = [np.array([0.15, 0.28], dtype=np.float32)]  # 设置尚未接触的真实起点
rollout = [truth[0].copy()]  # 连续预测只在这里接收一次真实状态
one_step = [truth[0].copy()]  # 单步对照从相同起点开始
for action in actions:  # 逐个处理相同的动作序列
    one_step.append(truth[-1] + predict_delta(np.r_[truth[-1], action]))  # 每一步重新使用真实输入
    truth.append(transition(truth[-1], action))  # 获得用于核对的真实下一状态
    rollout.append(rollout[-1] + predict_delta(np.r_[rollout[-1], action]))  # 将上一步预测继续作为输入
truth, rollout, one_step = map(np.asarray, (truth, rollout, one_step))  # 整理成便于画图的数组
fig, ax = plt.subplots(figsize=(7, 3.5))  # 创建三种物体位置的对照图
ax.plot(truth[:, 1], color="black", label="Recorded truth")  # 黑线表示教学环境的真实结果
ax.plot(one_step[:, 1], linestyle=":", label="One-step prediction")  # 点线表示每步用真实输入的预测
ax.plot(rollout[:, 1], linestyle="--", label="Recursive prediction")  # 虚线表示连续使用自身结果的预测
ax.set(xlabel="Step", ylabel="Object position (m)")  # 标明时间与物体位置
ax.legend()  # 标明三种计算方式
plt.show()  # 显示预测与真实结果的差别
for horizon_step in (1, 3, 5, 10, 30):  # 选择多个预测距离观察误差怎样变化
    horizon_error = abs(rollout[horizon_step, 1] - truth[horizon_step, 1]) * 1000  # 将对应时刻的物体位置误差换算成毫米
    print(f"连续预测到第 {horizon_step:2d} 步时，物体位置误差为 {horizon_error:.2f} mm")  # 输出不同预测距离的误差
assert np.isfinite(rollout).all()  # 检查连续预测没有产生无效数字
assert object_mae < baseline_mae  # 检查模型至少优于本实验的不动基线


### 最后一组结果要分三层阅读

第一层看接触与未接触误差。未接触时物体本来就不动，因此“不动基线”很容易得到低误差；接触样本才真正检验网络是否学到动作造成的物体变化。全部样本平均值不能替代这两项分开统计。

第二层看同一状态下的三种动作。向左动作的推杆变化应为负，大幅向右动作应比小幅向右产生更大的变化；物体是否移动还取决于当前间隙。如果动作改变而预测几乎不变，需要检查网络是否忽略了动作输入。

第三层再读图。黑线是真实记录，点线每一步重新使用真实状态，只反映一步预测；虚线把上一步模型结果继续作为输入，因此会携带先前误差。打印的第 1、3、5、10、30 步误差与虚线偏离黑线的程度相对应。

本实验仍是简化的一维接触模型，没有实现多步训练，也不包含摩擦和旋转。它验证的是第 14 章最核心的三件事：动作与前后状态怎样对齐、一步模型怎样训练、连续使用预测为什么更难。
